# Week 08 — Python Solution Lab
## Centre of Mass & Static Equilibrium

**Companion to `notebooks/Week_08.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_08.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P1` | Centre of Mass of Three Point Masses | `np.average` weighted, defining property |
| **L2 · Intermediate** | `P6` | Ladder Against a Smooth Wall | 3×3 equilibrium solve, pivot-independence check |
| **L3 · Challenge** | `P9` | Two-Segment Robot Arm Joint Torques | forward kinematics + motor-sizing sweep |

---

## L1 · Basic — P1: Centre of Mass of Three Point Masses

> **Problem (Week_08.ipynb, L1 — P1).** Three point masses: $m_1 = 1.0$ kg at $(0,0)$,
> $m_2 = 3.0$ kg at $(4.0, 0)$ m, $m_3 = 2.0$ kg at $(2.0, 3.0)$ m. Find the centre of mass.

**Diagram → Principle.** The centre of mass is the mass-weighted average position — the balance
point of the system.

**Equation.** $\vec r_{\rm cm} = \dfrac{\sum m_i \vec r_i}{\sum m_i}$.

**Hand prediction.** $x_{\rm cm} = (0 + 12 + 4)/6 = 2.67$ m, $y_{\rm cm} = (0+0+6)/6 = 1.00$ m.

**What Python adds.** Written with `np.average(..., weights=m)` the formula becomes one line that
works in any number of dimensions for any number of masses. We then verify the defining property
directly — that $\sum m_i(\vec r_i - \vec r_{\rm cm}) = \vec 0$ — and plot the configuration so
the "balance point" reading is visible rather than abstract.

In [ ]:
# ═══ W08 · L1 · P1 — Centre of mass as a weighted average ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
m = np.array([1.0, 3.0, 2.0])                         # kg
r = np.array([[0.0, 0.0], [4.0, 0.0], [2.0, 3.0]])    # m

# --- PREDICT: one line, any dimension, any number of masses -------------
r_cm = np.average(r, axis=0, weights=m)
print(f"total mass M = {m.sum():.1f} kg")
print(f"r_cm = ({r_cm[0]:.4f}, {r_cm[1]:.4f}) m")

# --- VERIFY 1: the long-hand sum ----------------------------------------
r_cm_manual = (m[:, None] * r).sum(axis=0) / m.sum()
print(f"long-hand sum(m_i r_i)/M = ({r_cm_manual[0]:.4f}, {r_cm_manual[1]:.4f}) m -> agrees")
assert np.allclose(r_cm, r_cm_manual)

# --- VERIFY 2: the DEFINING property, sum m_i (r_i - r_cm) = 0 ----------
residual = (m[:, None] * (r - r_cm)).sum(axis=0)
print(f"\nsum m_i (r_i - r_cm) = {residual}  -> zero, so it really is the balance point")
assert np.allclose(residual, 0.0)

# --- Plot ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5.4, 4.6))
ax.scatter(r[:, 0], r[:, 1], s=m*160, color="#1565c0", zorder=3, label="point masses")
for (x, y), mi in zip(r, m):
    ax.annotate(f"{mi:.0f} kg", (x, y), textcoords="offset points",
                xytext=(9, 7), fontsize=10)
    ax.plot([x, r_cm[0]], [y, r_cm[1]], color="grey", lw=.8, ls=":")
ax.plot(*r_cm, "*", color="#e65100", ms=22, zorder=4,
        label=f"CM ({r_cm[0]:.2f}, {r_cm[1]:.2f})")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.set_aspect("equal")
ax.grid(alpha=.3); ax.legend()
ax.set_title("W08 P1 — centre of mass pulled toward the 3 kg mass")
plt.tight_layout(); plt.show()

print(f"\nNote the CM sits at x = {r_cm[0]:.2f} m, right of the midpoint x = 2.0 m,")
print("because the 3 kg mass on the right outweighs the 1 kg on the left.")

# --- CHECK --------------------------------------------------------------
assert np.allclose(r_cm, [2.6667, 1.0000], atol=1e-3)
print(f"[OK] Matches textbook answer: ({r_cm[0]:.2f}, {r_cm[1]:.2f}) m")

## L2 · Intermediate — P6: Ladder Against a Smooth Wall

> **Problem (Week_08.ipynb, L2 — P6).** A $4.0$ m, $12$ kg ladder leans against a **frictionless**
> wall at $65^\circ$ to the floor. A $75$ kg person stands $3.0$ m up the ladder. Find the normal
> force from the wall and the friction force at the floor.

**Diagram → Principle.** Three equilibrium equations: $\sum F_x = 0$, $\sum F_y = 0$,
$\sum \tau = 0$. Choose the torque axis at the floor contact to eliminate two unknowns at once.

**Equation.** Torques about the base:
$N_w L\sin\theta = m_\ell g\frac{L}{2}\cos\theta + m_p g\, d\cos\theta$.

**Hand prediction.** $N_w = \dfrac{(12)(9.81)(2.0) + (75)(9.81)(3.0)}{4.0\tan 65^\circ} = 284.8$ N,
and since the wall is smooth, $f = N_w$.

> ⚠️ **Answer-key discrepancy.** The key printed in `Week_08.ipynb` gives $290.7$ N. With the
> problem's own $g = 9.81$ m/s² the correct value is $284.8$ N. The key is *approximately*
> reproduced by using $g \approx 10$ m/s² together with rounded trigonometric values — $g = 10$
> alone gives $290.28$ N, not $290.7$ N. The cell below computes both so the gap is explicit.

**What Python adds.** Two things. First, we solve the three equilibrium equations as a **linear
system** rather than by substitution, and then verify the torque balance about a *different*
pivot — if the answer is right, the choice of pivot cannot matter, and that is a genuinely
independent check. Second, the real question a ladder poses is *when does it slip*: we compute
the required $\mu_s$ and sweep the person's position to find how high they can safely climb.

In [ ]:
# ═══ W08 · L2 · P6 — Ladder equilibrium as a linear system, plus a slip analysis ═══
import numpy as np
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
L, m_lad, m_per, d, g = 4.0, 12.0, 75.0, 3.0, 9.81
theta = np.radians(65.0)
W_lad, W_per = m_lad*g, m_per*g

# Unknowns u = [N_wall, f_floor, N_floor]
# (1) sum Fx = 0:   N_wall - f_floor           = 0
# (2) sum Fy = 0:   N_floor - W_lad - W_per    = 0
# (3) sum tau about the BASE = 0:
#     N_wall * L sin(th)  -  W_lad*(L/2)cos(th)  -  W_per*d*cos(th) = 0
A = np.array([[1.0, -1.0, 0.0],
              [0.0,  0.0, 1.0],
              [L*np.sin(theta), 0.0, 0.0]])
b = np.array([0.0,
              W_lad + W_per,
              W_lad*(L/2)*np.cos(theta) + W_per*d*np.cos(theta)])
N_wall, f_floor, N_floor = np.linalg.solve(A, b)

print(f"weights: ladder {W_lad:.2f} N, person {W_per:.2f} N")
print(f"\nN_wall  = {N_wall:.2f} N")
print(f"f_floor = {f_floor:.2f} N   (equal to N_wall: the wall is frictionless)")
print(f"N_floor = {N_floor:.2f} N   (carries the entire weight)")

# --- VERIFY: torque balance about a DIFFERENT pivot (the top contact) ---
# Position vectors from the top of the ladder.
x_base, y_base = -L*np.cos(theta), -L*np.sin(theta)
tau_top = ( N_floor * x_base                       # normal at base
          - f_floor * y_base                       # friction at base (acts in -x)
          - W_lad  * (x_base/2)                    # ladder weight at mid-point
          - W_per  * (-(L - d)*np.cos(theta)) )    # person, measured from the top
print(f"\nnet torque about the TOP contact = {tau_top:.2e} N*m")
print("  -> zero for a different pivot too. The solution is genuinely in equilibrium.")
assert abs(tau_top) < 1e-9

# --- The real question: will it slip? -----------------------------------
mu_req = f_floor / N_floor
print(f"\nrequired floor friction coefficient mu_s >= f/N = {mu_req:.4f}")
print(f"  dry wood on concrete (mu ~ 0.5): {'SAFE' if mu_req < 0.5 else 'SLIPS'}")
print(f"  wet tile           (mu ~ 0.25): {'SAFE' if mu_req < 0.25 else 'SLIPS'}")

# --- Sweep: how high can the person climb? ------------------------------
ds  = np.linspace(0, L, 400)
Nw  = (W_lad*(L/2)*np.cos(theta) + W_per*ds*np.cos(theta)) / (L*np.sin(theta))
mus = Nw / (W_lad + W_per)

fig, ax = plt.subplots(figsize=(7.2, 4))
ax.plot(ds, mus, color="#1565c0", lw=2.5, label="$\\mu_s$ required")
for mu, lbl, col in ((0.50, "dry (0.50)", "#2e7d32"), (0.25, "wet tile (0.25)", "#e65100")):
    ax.axhline(mu, ls="--", color=col, label=lbl)
    if mus.max() > mu:
        d_lim = ds[np.argmax(mus > mu)]
        ax.plot(d_lim, mu, "o", color=col, ms=9, zorder=5)
        print(f"  with mu_s = {mu}: safe only up to {d_lim:.2f} m along the ladder "
              f"({100*d_lim/L:.0f}% of its length)")
ax.axvline(d, ls=":", c="grey", label=f"this problem: {d:.1f} m")
ax.set_xlabel("distance of the person along the ladder (m)")
ax.set_ylabel("$\\mu_s$ required at the floor")
ax.set_title("W08 P6 — the higher you climb, the more friction you need")
ax.grid(alpha=.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(N_wall - 284.76) < 0.05 and np.isclose(N_wall, f_floor)

# --- Reconciling with the printed answer key ----------------------------
Nw_g10 = (W_lad/g*10*(L/2) + W_per/g*10*d) * np.cos(theta) / (L*np.sin(theta))
print(f"\nANSWER-KEY NOTE")
print(f"  with g = 9.81 m/s^2 (as the problem states): N_wall = {N_wall:.2f} N")
print(f"  with g = 10.0 m/s^2                        : N_wall = {Nw_g10:.2f} N")
print(f"  the key prints 290.7 N. Using g ~ 10 approximately reproduces it, but not")
print(f"  exactly ({Nw_g10:.2f} N) -- the remaining gap is rounded trig values. Either")
print(f"  way it disagrees with the stated g = 9.81, which gives {N_wall:.2f} N.")
assert abs(Nw_g10 - 290.3) < 0.5

print(f"\n[OK] N_wall = f_floor = {N_wall:.2f} N (g = 9.81 m/s^2)")

## L3 · Challenge — P9: Two-Segment Robot Arm Joint Torques

> **Problem (Week_08.ipynb, L3 — P9).** A robot arm: segment 1 ($L_1 = 0.40$ m, $m_1 = 1.5$ kg)
> at $\theta_1 = 30^\circ$ above horizontal; segment 2 ($L_2 = 0.30$ m, $m_2 = 0.80$ kg) hangs
> vertically from the elbow, carrying a $2.0$ kg payload at its tip. Each segment's mass acts at
> its geometric centre. Find the torque at the elbow and at the shoulder.

**Diagram → Principle.** Only the **horizontal lever arm** matters for a vertical gravity load.
Work outward-in: the elbow carries segment 2 and the payload; the shoulder carries everything.

**Equation.** $\tau = \sum_i m_i g\, x_i$, where $x_i$ is the horizontal offset from the joint.

**Hand prediction.** Segment 2 hangs straight down, so its weight and the payload act on the same
vertical line as the elbow — **zero** horizontal offset, hence zero elbow torque.

**What Python adds.** Building the arm from joint positions with a small forward-kinematics
helper means we can ask the question that actually matters for motor selection: *how does the
shoulder torque vary as the arm sweeps through its whole range?* The peak of that curve is the
motor spec, and it does not occur at the configuration the problem happens to name.

In [ ]:
# ═══ W08 · L3 · P9 — Robot arm joint torques, then a full sweep for motor sizing ═══
import numpy as np
import matplotlib.pyplot as plt

g = 9.81
L1, m1 = 0.40, 1.5      # shoulder link
L2, m2 = 0.30, 0.80     # forearm, hanging vertically
m_pay  = 2.0

def arm_geometry(th1_deg, th2_deg=-90.0):
    """Absolute joint/centre positions. th2 is measured from the +x axis."""
    t1, t2 = np.radians(th1_deg), np.radians(th2_deg)
    shoulder = np.array([0.0, 0.0])
    elbow    = shoulder + L1*np.array([np.cos(t1), np.sin(t1)])
    tip      = elbow    + L2*np.array([np.cos(t2), np.sin(t2)])
    c1       = shoulder + 0.5*L1*np.array([np.cos(t1), np.sin(t1)])   # link-1 centre
    c2       = elbow    + 0.5*L2*np.array([np.cos(t2), np.sin(t2)])   # link-2 centre
    return shoulder, elbow, tip, c1, c2

def joint_torques(th1_deg, th2_deg=-90.0):
    """Static gravity torque required at each joint (N*m, magnitude)."""
    sh, el, tip, c1, c2 = arm_geometry(th1_deg, th2_deg)
    # elbow: carries link 2 and the payload
    tau_el = m2*g*abs(c2[0] - el[0]) + m_pay*g*abs(tip[0] - el[0])
    tau_el = m2*g*(c2[0] - el[0]) + m_pay*g*(tip[0] - el[0])
    # shoulder: carries everything
    tau_sh = m1*g*(c1[0] - sh[0]) + m2*g*(c2[0] - sh[0]) + m_pay*g*(tip[0] - sh[0])
    return tau_el, tau_sh

# --- PREDICT at the stated configuration --------------------------------
sh, el, tip, c1, c2 = arm_geometry(30.0)
tau_el, tau_sh = joint_torques(30.0)
print("geometry at theta1 = 30 deg, forearm vertical:")
for nm, pt in (("shoulder", sh), ("elbow", el), ("tip", tip),
               ("link1 CoM", c1), ("link2 CoM", c2)):
    print(f"  {nm:10s} ({pt[0]:+.4f}, {pt[1]:+.4f}) m")

print(f"\n(a) elbow torque    = {tau_el:.4f} N*m")
print("     the forearm and payload hang on a vertical line THROUGH the elbow,")
print("     so their horizontal lever arm is zero. [that is why it vanishes]")
print(f"(b) shoulder torque = {tau_sh:.4f} N*m")

# --- VERIFY: same answer from the system centre of mass -----------------
M     = m1 + m2 + m_pay
r_cm  = (m1*c1 + m2*c2 + m_pay*tip) / M
tau_cm = M * g * r_cm[0]
print(f"\ncross-check via the system CoM at x = {r_cm[0]:.4f} m:")
print(f"  tau = M g x_cm = {tau_cm:.4f} N*m  -> agrees with the term-by-term sum")
assert np.isclose(tau_sh, tau_cm)
assert abs(tau_el) < 1e-12

# --- MOTOR SIZING: sweep the shoulder through its range -----------------
angles = np.linspace(-90, 90, 721)
taus   = np.array([joint_torques(a)[1] for a in angles])
i_pk   = np.argmax(np.abs(taus))
print(f"\nsweeping theta1 over [-90, +90] deg with the forearm held vertical:")
print(f"  peak shoulder torque = {abs(taus[i_pk]):.4f} N*m at theta1 = {angles[i_pk]:.1f} deg")
print(f"  the stated 30 deg pose needs only {tau_sh:.3f} N*m "
      f"({100*tau_sh/abs(taus[i_pk]):.0f}% of the peak)")
print("  -> size the motor for the horizontal pose, not the pose in the question.")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 4))
for a, alpha in ((0, .25), (30, 1.0), (60, .25), (-45, .25)):
    s, e, t_, _, _ = arm_geometry(a)
    ax1.plot([s[0], e[0], t_[0]], [s[1], e[1], t_[1]], "o-", lw=3, ms=7,
             color="#1565c0", alpha=alpha)
ax1.plot(0, 0, "s", color="k", ms=11)
ax1.set_aspect("equal"); ax1.grid(alpha=.3)
ax1.set_xlabel("x (m)"); ax1.set_ylabel("y (m)")
ax1.set_title("arm poses (solid = the 30 deg case)")

ax2.plot(angles, taus, color="#e65100", lw=2.5)
ax2.plot(angles[i_pk], taus[i_pk], "o", color="crimson", ms=9, zorder=5,
         label=f"peak {abs(taus[i_pk]):.2f} N*m")
ax2.plot(30, tau_sh, "o", color="#2e7d32", ms=9, zorder=5,
         label=f"this problem {tau_sh:.2f} N*m")
ax2.axhline(0, c="k", lw=.8)
ax2.set_xlabel("shoulder angle $\\theta_1$ (deg)"); ax2.set_ylabel("shoulder torque (N*m)")
ax2.set_title("what the shoulder motor must deliver")
ax2.grid(alpha=.3); ax2.legend(fontsize=9)
plt.suptitle("W08 P9 — two-segment arm, 2 kg payload", y=1.03)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(tau_el) < 1e-9, "elbow torque must vanish for a vertical forearm"
assert abs(tau_sh - 12.09) < 0.05, f"tau_sh = {tau_sh}"
print(f"\n[OK] elbow torque = 0 N*m (vertical forearm); shoulder torque = {tau_sh:.2f} N*m")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_08.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
